# DTP workflow demonstration

Ogan Ilir 2016 using DTP approach

In [1]:
import ee 
import luma_ge

#Option 1: Manual authenticate using personal account
#Instructions for manual authentication
# luma_ge.print_auth_instructions()
#uncomment the below line and follow earth engine authentication process
# luma_ge.authenticate_manually()

#Option 2: Autheticate using service account (json file)
#service_account_path = '../auth/earth-engine-451407-520c1ef64879.json' #(if failed use this)
service_account_path = '../auth/ee-epstm2024.json'
luma_ge.initialize_with_service_account(service_account_path)

Service account initialization failed: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.


True

# 1. Acquisition of Near-Cloud-Free Satellite Imagery

In [2]:
import geemap
from luma_ge.data_acquisition import Reflectance_Data, final_Image

### Area of Interest Definition

In [3]:
aoi = geemap.shp_to_ee("../data/modular_mapping_approach/oganilir_modular_td/AOI_Oganilir.shp")

In [4]:
#========== FIRST RETRIVE THE MULTISPECTRAL BAND===========
#Intialize the relfectance class data function
optical_reflectance = Reflectance_Data()
#Initialize the final image class for composite creation
composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2017-01-01'
end = '2017-12-31'
#get the image collection and corresponding statistics
landsat_data, meta = optical_reflectance.get_optical_data(aoi, start, end, optical_data='L8_SR', 
                                                           cloud_cover=40, compute_detailed_stats=False)
#create mosaic between image collection, and clip based on AOI
mosaic_landsat = composite.get_quality_mosaic(landsat_data, aoi, quality_band= 'NDVI', calculate_coverage=False) #REPLACE OLD CODE WITH THE NEW ONE HERE
#Alternatively you can use temporal aggregation (ee reducer) to create mode cloudless imagery
#Add new functionality to calculate the coverage of the composite
median_landsat, coverage = composite.get_temporal_composite(landsat_data, aoi, reducer='Median', calculate_coverage=True) #REPLACE OLD CODE WITH THE NEW ONE HERE
#visualization parameter
l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['NIR', 'RED', 'GREEN']}

#retive thermal bands from TOA
thermal_bands, thermal_stats = optical_reflectance.get_thermal_bands(aoi, start, end, cloud_cover=40, thermal_data='L8_TOA', compute_detailed_stats=False)
median_thermal = composite.get_temporal_composite(thermal_bands, aoi, reducer='Median') #REPLACE THE OLD CODE WITH THE NEW ONE
thermal_vis = {'min': 286,'max': 300,'gammma': 0.4}
#stacked all landsat bands and convert to float(making sure all data type are compatible)
stacked_landsat = median_landsat.addBands(median_thermal).toFloat()


2026-05-27 15:47:52,904 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-05-27 15:47:52,905 - final_Image - INFO - final_Image creation initialized.
2026-05-27 15:47:52,907 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2026-05-27 15:47:52,907 - Reflectance_Data - INFO - Date range: 2017-01-01 to 2017-12-31
2026-05-27 15:47:52,908 - Reflectance_Data - INFO - Cloud cover threshold: 40%
2026-05-27 15:47:52,909 - Reflectance_Data - INFO - detailed statistics will not be computed
2026-05-27 15:47:52,910 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-05-27 15:47:52,912 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2026-05-27 15:48:04,709 - final_Image - INFO - Creating quality mosaic from 11 images using NDVI as quality metric
2026-05-27 15:48:04,715 - final_Image - INFO - Quality mosaic created covering AOI with best available pixels
202

# 2. Land-cover classification Scheme
Upload the classification scheme csv
Currently this defined object in this chunk is not being used in the final classification workflow but will be integrated later in the process of building the decision tree

In [ ]:
# from luma_ge.classification_scheme import LULC_Scheme_Manager
# import pandas as pd
# #Initialize the LULC Scheme Manager
# manager = LULC_Scheme_Manager()
# #path to csv 
# csv_path = "../data/modular_mapping_approach/oganilir_modular_td/rplus_classification_scheme.csv"
# # Load the CSV
# df = pd.read_csv(csv_path, sep=None, engine="python")
# print(df)
# # Auto-detect columns
# id_col, name_col, color_col = manager.auto_detect_csv_columns(df)
# print(f"\nAuto-detected columns:")
# print(f"ID column: {id_col}")
# print(f"Name column: {name_col}")
# print(f"Color column: {color_col}")

   ﻿id_rplus               name_rplus
0          2       Logged-over forest
1         23                Waterbody
2         20               Settlement
3          9     Oil Palm monoculture
4         21             Cleared land
5         13                 Cropland
6          4  Tree-based not oil palm
7         18        Grass and savanna

Auto-detected columns:
ID column: ﻿id_rplus
Name column: name_rplus
Color column: None


In [ ]:
# manager.process_csv_upload(df, id_col, name_col, color_col)
# classification_df = manager.get_dataframe()


# 3. Upload modular reference data
Use modular training dataset workflow

In [7]:
import pandas as pd
import numpy as np

modular_td_csv = '../data/modular_mapping_approach/oganilir_modular_td/modular_td_oganilir_v6.csv'
modular_td = pd.read_csv(modular_td_csv)

# 4. Generate primitive stack
For demonstration purpose, only four primitives will be used:
- Tree cover
- Tree height
- Built presence
- Water presence

## 4.1 Choose the primitives from the modular training data

In [8]:
chosen_primitives = ['tree_cover',
                      'vegetation_height',
                      'builtup_presence',
                      'waterbody_presence']

# keep the first 4 columns (id, longitude, latitude, label) and the chosen primitives
df_train = modular_td[
    list(modular_td.columns[:4]) + chosen_primitives
]

for col in chosen_primitives:
    df_train[col] = pd.to_numeric(df_train[col], errors='coerce')

df_train[chosen_primitives] = (
    df_train[chosen_primitives]
    .fillna(0)
)

# Convert dataframe rows into Earth Engine Features
features = []

for _, row in df_train.iterrows():
    
    # Create geometry from latitude & longitude
    point = ee.Geometry.Point([row['longitude'], row['latitude']])
    
    # Convert row to dictionary
    props = row.to_dict()
    
    # Create feature
    feature = ee.Feature(point, props)
    
    features.append(feature)

# Create FeatureCollection
df_train_fc = ee.FeatureCollection(features)

print(df_train_fc.first().getInfo())

C:\Users\widijanto\AppData\Local\Temp\ipykernel_27016\3221434499.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
C:\Users\widijanto\AppData\Local\Temp\ipykernel_27016\3221434499.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train[chosen_primitives] = (


{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [104.5638596, -3.622319527]}, 'id': '0', 'properties': {'builtup_presence': 0, 'class_id': 2, 'class_name': 'Secondary Dryland Forest', 'latitude': -3.622319527, 'longitude': 104.5638596, 'tree_cover': 35, 'vegetation_height': 10, 'waterbody_presence': 0}}


## 4.2 Feature extraction

In [9]:
training_sample = stacked_landsat.sampleRegions(
    collection=df_train_fc,
    properties=chosen_primitives,
    scale=30,
    geometries=False
)

## 4.3 Run random forest for each primitive layer

In [10]:
band_names = stacked_landsat.bandNames()

def train_primitive(primitive):
    clf = (ee.Classifier.smileRandomForest(50)
             .setOutputMode('REGRESSION') # important!
             .train(features=training_sample, classProperty=primitive, inputProperties=band_names))
    return stacked_landsat.classify(clf).rename(primitive)

primitive_images = [train_primitive(p) for p in chosen_primitives]
primitive_stack = ee.Image.cat(primitive_images)

### Quick value check of the primitive stack

In [11]:
for primitive in chosen_primitives:
    
    stats = primitive_stack.select(primitive).reduceRegion(
        reducer=ee.Reducer.minMax()
                  .combine(ee.Reducer.mean(), sharedInputs=True)
                  .combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=aoi,
        scale=30,
        maxPixels=1e13
    )
    
    print(f"\n===== {primitive} =====")
    print(stats.getInfo())


===== tree_cover =====
{'tree_cover_max': 63.49333333333333, 'tree_cover_mean': 26.60666568562698, 'tree_cover_min': 1.25, 'tree_cover_stdDev': 10.844009918748121}

===== vegetation_height =====
{'vegetation_height_max': 14.896666666666668, 'vegetation_height_mean': 4.84186726130786, 'vegetation_height_min': 0, 'vegetation_height_stdDev': 3.6786969790361437}

===== builtup_presence =====
{'builtup_presence_max': 0.967389, 'builtup_presence_mean': 0.0918881614708624, 'builtup_presence_min': 0, 'builtup_presence_stdDev': 0.13640885212000312}

===== waterbody_presence =====
{'waterbody_presence_max': 79.10666666666668, 'waterbody_presence_mean': 5.549338374262009, 'waterbody_presence_min': 0, 'waterbody_presence_stdDev': 7.937495309624026}


# 5. Set ruleset
The decision tree was built manually
The function adapted from RLCMS technical guide

In [12]:
class_struct = {
    'undisturbedForest': {'number': 1,  'color': '#0B6623'},
    'loggedOverForest':  {'number': 2,  'color': '#6B8E23'},
    'cropland':          {'number': 13, 'color': '#C827D7'},
    'grassAndSavanna':   {'number': 18, 'color': '#C2D26B'},
    'settlement':        {'number': 19, 'color': '#FC0D00'},
    'shrub':             {'number': 20, 'color': '#8FBC8F'},
    'clearedLand':       {'number': 21, 'color': '#D2B48C'},
    'waterBody':         {'number': 23, 'color': '#2B65EC'},
}

node_struct = {
    'ROOT': {'band': 'waterbody_presence', 'threshold': 13,   'left': 'terminal', 'leftName': 'waterBody',         'right': 'SATU'},
    'SATU': {'band': 'builtup_presence',   'threshold': 0.25, 'left': 'terminal', 'leftName': 'settlement',        'right': 'DUA'},
    'DUA':  {'band': 'tree_cover',         'threshold': 30,   'left': 'EMPAT',                                     'right': 'TIGA'},
    'TIGA': {'band': 'builtup_presence',   'threshold': 0.25, 'left': 'terminal', 'leftName': 'clearedLand',       'right': 'LIMA'},
    'EMPAT':{'band': 'vegetation_height',  'threshold': 6,    'left': 'ENAM',                                      'right': 'terminal', 'rightName': 'shrub'},
    'LIMA': {'band': 'vegetation_height',  'threshold': 6,    'left': 'terminal', 'leftName': 'cropland',          'right': 'terminal', 'rightName': 'grassAndSavanna'},
    'ENAM': {'band': 'vegetation_height',  'threshold': 11,   'left': 'terminal', 'leftName': 'undisturbedForest', 'right': 'terminal', 'rightName': 'loggedOverForest'},
}

def build_decision_tree(node_struct, class_struct, id_, node, dt_lines):
    lnode, rnode = 2 * node, 2 * node + 1
    d = node_struct[id_]
    band, thr = d['band'], d['threshold']
    left, right = d.get('left'), d.get('right')

    if left == 'terminal':
        ln = class_struct[d['leftName']]['number']
        dt_lines.append(f"{lnode}) {band}>={thr} 9999 9999 {ln} *")
        if right == 'terminal':
            rn = class_struct[d['rightName']]['number']
            dt_lines.append(f"{rnode}) {band}<{thr} 9999 9999 {rn} *")
        else:
            dt_lines.append(f"{rnode}) {band}<{thr} 9999 9999 9999")
            build_decision_tree(node_struct, class_struct, right, rnode, dt_lines)
    else:
        dt_lines.append(f"{lnode}) {band}>={thr} 9999 9999 9999")
        build_decision_tree(node_struct, class_struct, left, lnode, dt_lines)
        if right == 'terminal':
            rn = class_struct[d['rightName']]['number']
            dt_lines.append(f"{rnode}) {band}<{thr} 9999 9999 {rn} *")
        else:
            dt_lines.append(f"{rnode}) {band}<{thr} 9999 9999 9999")
            build_decision_tree(node_struct, class_struct, right, rnode, dt_lines)
    return dt_lines

dt_lines = ['1) root 9999 9999 9999']
decision_tree = '\n'.join(build_decision_tree(node_struct, class_struct, 'ROOT', 1, dt_lines))
print('Decision tree:\n', decision_tree)



Decision tree:
 1) root 9999 9999 9999
2) waterbody_presence>=13 9999 9999 23 *
3) waterbody_presence<13 9999 9999 9999
6) builtup_presence>=0.25 9999 9999 19 *
7) builtup_presence<0.25 9999 9999 9999
14) tree_cover>=30 9999 9999 9999
28) vegetation_height>=6 9999 9999 9999
56) vegetation_height>=11 9999 9999 1 *
57) vegetation_height<11 9999 9999 2 *
29) vegetation_height<6 9999 9999 20 *
15) tree_cover<30 9999 9999 9999
30) builtup_presence>=0.25 9999 9999 21 *
31) builtup_presence<0.25 9999 9999 9999
62) vegetation_height>=6 9999 9999 13 *
63) vegetation_height<6 9999 9999 18 *


# 6. Generate LULC Map

In [13]:
land_class = primitive_stack.classify(ee.Classifier.decisionTree(decision_tree))


In [14]:
class_colors = ['#0B6623','#6B8E23','#C827D7','#C2D26B','#FC0D00','#8FBC8F','#D2B48C','#2B65EC']

remapped_vis = land_class.remap(
    [1, 2, 13, 18, 19, 20, 21, 23],
    [0, 1,  2,  3,  4,  5,  6,  7]
)

Map = geemap.Map()
Map.centerObject(aoi, 8)
Map.addLayer(remapped_vis, {'min': 0, 'max': 7, 'palette': class_colors}, 'DT Classification (visual)')
Map.addLayer(land_class,   {'min': 1, 'max': 23},                         'DT Classification (raw)', False)

legend_dict = {
    'Primary Dryland Forest': '#0B6623',
    'Secondary Dryland Forest': '#6B8E23',
    'Other Cropland': '#C827D7',
    'Grass Savanna': '#C2D26B',
    'Settlement': '#FC0D00',
    'Shrub': '#8FBC8F',
    'Cleared Land': '#D2B48C',
    'Water Body': '#2B65EC'
}

Map.add_legend(
    title='Land Cover Classification',
    legend_dict=legend_dict
)

Map

Map

Map(center=[-3.4152616959981676, 104.60534276364243], controls=(WidgetControl(options=['position', 'transparen…

# 5a Set ruleset of second scheme

In [15]:
class_struct_2 = {
    'primaryDrylandForest':    {'number': 1,  'color': '#004d00'},
    'secondaryDrylandForest':  {'number': 2,  'color': '#3a7a3a'},
    'primaryMangroveForest':   {'number': 3,  'color': '#005f40'},
    'secondaryMangroveForest': {'number': 4,  'color': '#4a8c6f'},
    'primarySwampForest':      {'number': 5,  'color': '#1a5c2e'},
    'secondarySwampForest':    {'number': 6,  'color': '#5a9a6a'},
    'plantationForest':        {'number': 7,  'color': '#8db87a'},
    'rubberMonoculture':       {'number': 8,  'color': '#c8a400'},
    'oilPalmMonoculture':      {'number': 9,  'color': '#e6c619'},
    'cacaoMonoculture':        {'number': 10, 'color': '#7b4f2e'},
    'coconutMonoculture':      {'number': 11, 'color': '#d4a96a'},
    'otherMonoculture':        {'number': 12, 'color': '#c8b87a'},
    'otherCropland':           {'number': 13, 'color': '#f5d57a'},
    'coffeeAgroforestry':      {'number': 14, 'color': '#6b3a2a'},
    'rubberAgroforestry':      {'number': 15, 'color': '#b89a50'},
    'mixedHomeGarden':         {'number': 16, 'color': '#d4c87a'},
    'paddyField':              {'number': 17, 'color': '#a8d48a'},
    'grassSavanna':            {'number': 18, 'color': '#c2d26b'},
    'shrub':                   {'number': 19, 'color': '#8fbc8f'},
    'settlement':              {'number': 20, 'color': '#fc0d00'},
    'clearedLand':             {'number': 21, 'color': '#d2b48c'},
    'miningArea':              {'number': 22, 'color': '#8c7a6b'},
    'waterBody':               {'number': 23, 'color': '#2b65ec'},
    'fishPond':                {'number': 24, 'color': '#6aaad4'},
}

# ── NODE STRUCTURE ────────────────────────────────────────────────────────────

# =============================================================================

node_struct_2 = {

    # ── LEVEL 1: Water mask ───────────────────────────────────────────────────
    'ROOT': {
        'band': 'waterbody_presence', 'threshold': 40,
        'left': 'terminal', 'leftName': 'waterBody',   # ≥ 40 → open water
        'right': 'BUILT'                               # < 40 → land
    },

    # ── LEVEL 2: Built-up ────────────────────────────────────────────────────
    'BUILT': {
        'band': 'builtup_presence', 'threshold': 0.9,
        'left': 'terminal', 'leftName': 'settlement',  # ≥ 0.9 → settlement
        'right': 'WATER_TRANS'
    },

    # ── LEVEL 3: Transitional wet zone ───────────────────────────────────────
    'WATER_TRANS': {
        'band': 'waterbody_presence', 'threshold': 10,
        'left': 'WET_TREE',                            # ≥ 10 → wet/flooded
        'right': 'DRY_TREE'                            # < 10 → dryland
    },

    # ── LEVEL 4 (WET): Tree cover ────────────────────────────────────────────
    'WET_TREE': {
        'band': 'tree_cover', 'threshold': 45,
        'left': 'WET_TALL',
        'right': 'WET_OPEN'
    },

    # ── Dense wet vegetation ─────────────────────────────────────────────────
    'WET_TALL': {
        'band': 'vegetation_height', 'threshold': 10,
        'left': 'terminal', 'leftName': 'primarySwampForest',
        'right': 'terminal', 'rightName': 'secondarySwampForest'
    },

    # ── Open / transitional wet vegetation ──────────────────────────────────
    'WET_OPEN': {
        'band': 'tree_cover', 'threshold': 20,
        'left': 'terminal', 'leftName': 'secondaryMangroveForest',
        'right': 'WET_LOW'
    },

    'WET_LOW': {
        'band': 'vegetation_height', 'threshold': 1,
        'left': 'terminal', 'leftName': 'fishPond',
        'right': 'terminal', 'rightName': 'waterBody'
    },

    # ── LEVEL 4 (DRY): Tree cover ────────────────────────────────────────────
    'DRY_TREE': {
        'band': 'tree_cover', 'threshold': 45,
        'left': 'DRY_TALL',
        'right': 'DRY_OPEN'
    },

    # ── Dense dry vegetation ─────────────────────────────────────────────────
    'DRY_TALL': {
        'band': 'vegetation_height', 'threshold': 10,
        'left': 'terminal', 'leftName': 'primaryDrylandForest',
        'right': 'DRY_MEDIUM'
    },

    # ── Medium-height dense canopy ───────────────────────────────────────────
    'DRY_MEDIUM': {
        'band': 'vegetation_height', 'threshold': 5,
        'left': 'terminal', 'leftName': 'secondaryDrylandForest',
        'right': 'terminal', 'rightName': 'plantationForest'
    },

    # ── Moderate canopy cover ────────────────────────────────────────────────
    'DRY_OPEN': {
        'band': 'tree_cover', 'threshold': 20,
        'left': 'OPEN_HEIGHT',
        'right': 'OPEN_LAND'
    },

    # ── Medium canopy systems ────────────────────────────────────────────────
    'OPEN_HEIGHT': {
        'band': 'vegetation_height', 'threshold': 6,
        'left': 'terminal', 'leftName': 'rubberAgroforestry',
        'right': 'LOW_TREE'
    },

    # ── Low tree systems ─────────────────────────────────────────────────────
    'LOW_TREE': {
        'band': 'vegetation_height', 'threshold': 3,
        'left': 'terminal', 'leftName': 'mixedHomeGarden',
        'right': 'terminal', 'rightName': 'otherCropland'
    },

    # ── Open land ────────────────────────────────────────────────────────────
    'OPEN_LAND': {
        'band': 'vegetation_height', 'threshold': 2,
        'left': 'OPEN_SHORT',
        'right': 'terminal', 'rightName': 'clearedLand'
    },

    # ── Low vegetation ───────────────────────────────────────────────────────
    'OPEN_SHORT': {
        'band': 'tree_cover', 'threshold': 5,
        'left': 'terminal', 'leftName': 'grassSavanna',
        'right': 'terminal', 'rightName': 'shrub'
    },
}

dt_lines_2 = ['1) root 9999 9999 9999']
decision_tree_2 = '\n'.join(build_decision_tree(node_struct_2, class_struct_2, 'ROOT', 1, dt_lines_2))
print('Decision tree:\n', decision_tree_2)

Decision tree:
 1) root 9999 9999 9999
2) waterbody_presence>=40 9999 9999 23 *
3) waterbody_presence<40 9999 9999 9999
6) builtup_presence>=0.9 9999 9999 20 *
7) builtup_presence<0.9 9999 9999 9999
14) waterbody_presence>=10 9999 9999 9999
28) tree_cover>=45 9999 9999 9999
56) vegetation_height>=10 9999 9999 5 *
57) vegetation_height<10 9999 9999 6 *
29) tree_cover<45 9999 9999 9999
58) tree_cover>=20 9999 9999 4 *
59) tree_cover<20 9999 9999 9999
118) vegetation_height>=1 9999 9999 24 *
119) vegetation_height<1 9999 9999 23 *
15) waterbody_presence<10 9999 9999 9999
30) tree_cover>=45 9999 9999 9999
60) vegetation_height>=10 9999 9999 1 *
61) vegetation_height<10 9999 9999 9999
122) vegetation_height>=5 9999 9999 2 *
123) vegetation_height<5 9999 9999 7 *
31) tree_cover<45 9999 9999 9999
62) tree_cover>=20 9999 9999 9999
124) vegetation_height>=6 9999 9999 15 *
125) vegetation_height<6 9999 9999 9999
250) vegetation_height>=3 9999 9999 16 *
251) vegetation_height<3 9999 9999 13 *
63)

In [16]:
land_class_2 = primitive_stack.classify(ee.Classifier.decisionTree(decision_tree_2))


# 6a Generate LULC map second scheme

In [17]:
# ── VISUALIZATION ────────────────────────────────────────────────────────────

class_values = [
    1,   # primaryDrylandForest
    2,   # secondaryDrylandForest
    4,   # secondaryMangroveForest
    5,   # primarySwampForest
    6,   # secondarySwampForest
    7,   # plantationForest
    15,  # rubberAgroforestry
    16,  # mixedHomeGarden
    13,  # otherCropland
    18,  # grassSavanna
    19,  # shrub
    20,  # settlement
    21,  # clearedLand
    22,  # miningArea
    23,  # waterBody
    24   # fishPond
]

class_names = [
    'Primary Dryland Forest',
    'Secondary Dryland Forest',
    'Secondary Mangrove Forest',
    'Primary Swamp Forest',
    'Secondary Swamp Forest',
    'Plantation Forest',
    'Rubber Agroforestry',
    'Mixed Home Garden',
    'Other Cropland',
    'Grass Savanna',
    'Shrub',
    'Settlement',
    'Cleared Land',
    'Mining Area',
    'Water Body',
    'Fish Pond'
]

class_colors = [
    '#004d00',
    '#3a7a3a',
    '#4a8c6f',
    '#1a5c2e',
    '#5a9a6a',
    '#8db87a',
    '#b89a50',
    '#d4c87a',
    '#f5d57a',
    '#c2d26b',
    '#8fbc8f',
    '#fc0d00',
    '#d2b48c',
    '#8c7a6b',
    '#2b65ec',
    '#6aaad4'
]

# ── Remap sequentially for visualization ─────────────────────────────────────

remapped_vis = land_class_2.remap(
    class_values,
    list(range(len(class_values)))
)

# ── Create map ───────────────────────────────────────────────────────────────

Map = geemap.Map()
Map.centerObject(aoi, 8)

Map.addLayer(
    remapped_vis,
    {
        'min': 0,
        'max': len(class_values) - 1,
        'palette': class_colors
    },
    'DT Classification (visual)'
)

Map.addLayer(
    land_class_2,
    {'min': 1, 'max': 24},
    'DT Classification (raw)',
    False
)

# ── Legend ───────────────────────────────────────────────────────────────────

legend_dict = dict(zip(class_names, class_colors))

Map.add_legend(
    title='Land Cover Classification',
    legend_dict=legend_dict
)

Map

Map(center=[-3.4152616959981676, 104.60534276364243], controls=(WidgetControl(options=['position', 'transparen…